# BE-HPG — Phase 1: Data Preparation (MSD Spleen)

Preprocess **Task09 Spleen (CT)** into 2D axial slices for few-shot training. Everything is
saved to Google Drive so a disconnect never forces a re-download.

**How to run**
1. Runtime can be **CPU** (no GPU needed here) — but GPU runtime is fine too.
2. Run every cell top-to-bottom **once with `SMOKE = True`** (dry run on 2 volumes, ~1–2 min).
3. If it looks good, set **`SMOKE = False`** in the CONFIG cell and re-run cells 5–10
   (the download is cached, so only preprocessing repeats; ~15–25 min full).

Pipeline: load NIfTI → reorient canonical → binarize spleen → keep slices with ≥1% foreground
→ HU window [-125,275] → normalize [0,1] → resize 224 → save `.npz` + manifest CSV to Drive.

In [ ]:
# 1) Dependencies (Colab already has numpy/scipy/pandas/matplotlib/tqdm)
!pip -q install nibabel scikit-image

In [ ]:
# 2) Mount Drive + create the project folders
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/be-hpg'
for sub in ['downloads', 'raw', 'data/spleen']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
print('Drive ready at', DRIVE_ROOT)

In [ ]:
# 3) CONFIG  --  run with SMOKE=True first, then set SMOKE=False and re-run cells 5-10
SMOKE       = True
TASK        = 'Task09_Spleen'
FG_LABELS   = [1]              # spleen
HU          = (-125, 275)      # abdominal CT window, then normalize to [0,1]
MIN_FG      = 0.01             # keep axial slices with >= 1% foreground
SIZE        = 224
SPLIT_FRACS = {'train': 0.60, 'val': 0.15, 'test': 0.25}   # split BY VOLUME (no leakage)
SPLIT_SEED  = 42
S3_URL      = f'https://msd-for-monai.s3-us-west-2.amazonaws.com/{TASK}.tar'
print('SMOKE =', SMOKE, '| TASK =', TASK)

In [ ]:
# 4) Download (resumable) + extract  ->  persisted on Drive
import urllib.request, tarfile
tar_path = f'{DRIVE_ROOT}/downloads/{TASK}.tar'

def _remote_size(u):
    try:
        with urllib.request.urlopen(urllib.request.Request(u, method='HEAD')) as r:
            return int(r.headers.get('Content-Length'))
    except Exception:
        return None

def download(u, dest):
    total = _remote_size(u)
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    if total and have == total:
        print('already downloaded:', dest); return
    req = urllib.request.Request(u)
    if have: req.add_header('Range', f'bytes={have}-')
    from tqdm.auto import tqdm
    bar = tqdm(total=total, initial=have, unit='B', unit_scale=True, desc=TASK)
    with urllib.request.urlopen(req) as r, open(dest, 'ab' if have else 'wb') as f:
        while True:
            b = r.read(1 << 20)
            if not b: break
            f.write(b); bar.update(len(b))
    bar.close()

download(S3_URL, tar_path)
extracted = f'{DRIVE_ROOT}/raw/{TASK}'
if not (os.path.isdir(extracted) and os.listdir(extracted)):
    with tarfile.open(tar_path) as t:
        try: t.extractall(f'{DRIVE_ROOT}/raw', filter='data')
        except TypeError: t.extractall(f'{DRIVE_ROOT}/raw')
print('extracted ->', extracted, '|', sorted(os.listdir(extracted))[:6])

In [ ]:
# 5) Preprocessing functions (mirror of the locally-tested src/data/preprocess.py)
import numpy as np
import nibabel as nib
from skimage.transform import resize as sk_resize

def load_volume(p):
    img = nib.as_closest_canonical(nib.load(p))
    return np.asanyarray(img.dataobj, dtype=np.float32)

def binarize(label, fg_labels):
    return np.isin(np.rint(label).astype(np.int64), list(fg_labels)).astype(np.uint8)

def ct_window(img2d, lo, hi):
    x = np.clip(img2d.astype(np.float32), lo, hi)
    return ((x - lo) / max(hi - lo, 1e-6)).astype(np.float32)

def resize2d(arr, size, is_mask):
    ds = arr.shape[0] > size or arr.shape[1] > size
    out = sk_resize(arr.astype(np.float32), (size, size), order=0 if is_mask else 1,
                    mode='edge', anti_aliasing=(not is_mask and ds), preserve_range=True)
    return (out >= 0.5).astype(np.uint8) if is_mask else out.astype(np.float32)

def process_volume(vol, lab, fg_labels, min_fg, hu, size):
    mvol = binarize(lab, fg_labels)
    imgs, masks, idxs = [], [], []
    for i in range(vol.shape[2]):              # axis 2 = axial after canonical reorient
        m2d = mvol[:, :, i]
        if m2d.mean() < min_fg:                # >= 1% foreground filter (native res)
            continue
        imgs.append(resize2d(ct_window(vol[:, :, i], hu[0], hu[1]), size, False).astype(np.float16))
        masks.append(resize2d(m2d, size, True))
        idxs.append(i)
    return imgs, masks, idxs

print('functions ready')

In [ ]:
# 6) Discover labeled volumes (imagesTr + labelsTr)
import glob
img_dir, lab_dir = f'{extracted}/imagesTr', f'{extracted}/labelsTr'
files = []
for ip in sorted(glob.glob(f'{img_dir}/*.nii.gz')):
    name = os.path.basename(ip)
    if name.startswith('.'):                 # skip macOS ._ resource forks
        continue
    lp = f'{lab_dir}/{name}'
    if os.path.exists(lp):
        files.append((ip, lp, name[:-7]))    # vol_id = filename without .nii.gz
print('labeled volumes found:', len(files))
if SMOKE:
    files = files[:2]
print('processing', len(files), 'volume(s)', '[SMOKE]' if SMOKE else '[FULL]')

In [ ]:
# 7) Preprocess all volumes -> stacked arrays + manifest, saved to Drive
import pandas as pd
from tqdm.auto import tqdm
imgs_all, masks_all, vol_ids, slice_idx, manifest = [], [], [], [], []
for ip, lp, vid in tqdm(files):
    imgs, masks, idxs = process_volume(load_volume(ip), load_volume(lp), FG_LABELS, MIN_FG, HU, SIZE)
    for im, m, si in zip(imgs, masks, idxs):
        manifest.append({'global_index': len(imgs_all), 'vol_id': vid, 'slice_idx': int(si)})
        imgs_all.append(im); masks_all.append(m); vol_ids.append(vid); slice_idx.append(int(si))

images = np.stack(imgs_all).astype(np.float16) if imgs_all else np.zeros((0, SIZE, SIZE), np.float16)
masks  = np.stack(masks_all).astype(np.uint8)  if masks_all else np.zeros((0, SIZE, SIZE), np.uint8)
print('kept slices:', len(images), '| images', images.shape, images.dtype, '| masks', masks.shape)

suffix = '_smoke' if SMOKE else ''
npz_path = f'{DRIVE_ROOT}/data/spleen/spleen{suffix}.npz'
np.savez_compressed(npz_path, images=images, masks=masks,
                    vol_ids=np.asarray(vol_ids), slice_idx=np.asarray(slice_idx, dtype=np.int32))
print('saved', npz_path, '(%.1f MB)' % (os.path.getsize(npz_path) / 1e6))

In [ ]:
# 8) Patient/volume-disjoint splits + manifest CSV
rng = np.random.default_rng(SPLIT_SEED)
uniq = sorted(set(vol_ids))
uniq = [uniq[i] for i in rng.permutation(len(uniq))]
n = len(uniq)
ntr = min(round(SPLIT_FRACS['train'] * n), n)
nva = min(round(SPLIT_FRACS['val'] * n), n - ntr)
tr, va = set(uniq[:ntr]), set(uniq[ntr:ntr + nva])
split = {v: ('train' if v in tr else 'val' if v in va else 'test') for v in uniq}

df = pd.DataFrame(manifest)
df['split'] = df['vol_id'].map(split)
man_path = f'{DRIVE_ROOT}/data/spleen/manifest{suffix}.csv'
df.to_csv(man_path, index=False)
print('manifest:', man_path)
if len(df):
    print('\nvolumes per split:'); print(df.groupby('split')['vol_id'].nunique())
    print('\nslices per split:');  print(df['split'].value_counts())

In [ ]:
# 9) Save 6 sample slice+mask overlays (bring this PNG back into paper/figures/)
import matplotlib.pyplot as plt
n_show = min(6, len(images))
sel = np.linspace(0, len(images) - 1, n_show).astype(int) if len(images) else []
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, j in zip(axes.ravel(), sel):
    ax.imshow(images[j].astype(np.float32), cmap='gray')
    ax.imshow(np.ma.masked_where(masks[j] == 0, masks[j]), cmap='autumn', alpha=0.5)
    ax.set_title(f'{vol_ids[j]}  z={slice_idx[j]}', fontsize=8); ax.axis('off')
for ax in axes.ravel()[n_show:]:
    ax.axis('off')
plt.tight_layout()
fig_path = f'{DRIVE_ROOT}/data/spleen/dataset_samples.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.show()
print('saved overlay figure:', fig_path)

## ✅ Done — what success looks like
- A printed **kept-slice count** (a few hundred to a few thousand for the full run).
- `data/spleen/spleen.npz` and `data/spleen/manifest.csv` saved on Drive, plus the overlay figure.

## Next
- If the **SMOKE** run looked right, set `SMOKE = False` in cell 3 and re-run cells 4–9
  (download is cached → only preprocessing repeats).
- **Bring back to the repo:** download `data/spleen/dataset_samples.png` and drop it in
  `paper/figures/`. The `.npz` + `manifest.csv` stay on Drive (training reads them there).
- **If anything fails:** copy the full error from the failing cell back to me.